# 6. Analise de Sensibilidade (Morris Screen) -- Soja no Parana

Roda o mesmo metodo de Morris do pipeline de milho (`util/SensitivityAnalyzer.py`), mas
usando a variante `SoyMorrisScreeningAnalyzerPR` (`util/SensitivityAnalyzer_soja_pr.py`)
que troca a cultura ativa para `soybean` / `Soybean_VanHeemst_1988` e o calendario para o
da soja no PR, mantendo toda a infraestrutura de amostragem/execucao/analise do WOFOST
reaproveitada sem alteracao.

O resultado (ranking de parametros mais sensiveis por cluster) e salvo tanto em CSV
(para inspecao) quanto em um **JSON de ranking por cluster**, que a etapa 7
(Optimization) carrega dinamicamente -- diferente do pipeline de milho, que tem esse
ranking fixado como dicionario no codigo (`WOFOSTOptimizer.CLUSTER_PARAMS`), aqui ainda
nao existe uma calibracao previa para soja/PR, entao o ranking tem que vir do resultado
desta propria etapa.


In [1]:
import glob
import json
import os
import sys

import pandas as pd

sys.path.append(os.path.join(os.getcwd(), 'util'))
from util.SensitivityAnalyzer_soja_pr import SoyMorrisScreeningAnalyzerPR, SoyNetCDFDataLoader
from util.utils import WOFOST_bounds
from util.utils_soja_pr import setup_paths_soja_pr

paths = setup_paths_soja_pr()
DEFAULT_BOUNDS = WOFOST_bounds("all")

N_POINTS_PER_CLUSTER = 30  # PR tem bem menos municipios que o dataset global de milho;
                            # ajuste para cima se sobrar tempo/dados por cluster.

nc_loader = SoyNetCDFDataLoader(paths['COMPLETO'], results_dir=paths['RESULTS'])
analyzer = SoyMorrisScreeningAnalyzerPR(DEFAULT_BOUNDS, paths, nc_loader)


Encontrados 392 arquivos NetCDF


In [2]:
paths

{'BASE': 'd:\\_py\\AgroIA_prod',
 'DATA': 'd:\\_py\\AgroIA_prod\\inputs\\data\\soja_pr',
 'COORDINATES': 'd:\\_py\\AgroIA_prod\\inputs\\data\\soja_pr\\coordinates_pr.xlsx',
 'COMPLETO': 'd:\\_py\\AgroIA_prod\\inputs\\data\\soja_pr\\completo',
 'AGRO': 'd:\\_py\\AgroIA_prod\\inputs\\data\\agro\\agro_soybean_pr.agro',
 'CROP': 'd:\\_py\\AgroIA_prod\\inputs\\data\\crop',
 'SOIL': 'd:\\_py\\AgroIA_prod\\inputs\\data\\soil\\ec3.soil',
 'WEATHER_RAW': 'D:\\_py\\Clima_AgroIA\\data-raw\\xavier-data',
 'RESULTS': 'd:\\_py\\AgroIA_prod\\output\\soja_pr\\Sensitivity Analysis',
 'OPTIMIZATION': 'd:\\_py\\AgroIA_prod\\output\\soja_pr\\Optimization'}

In [3]:
important_params = analyzer.run_analysis(n_points_per_cluster=N_POINTS_PER_CLUSTER)
important_params


=== INICIANDO ANÁLISE DE SENSIBILIDADE (MORRIS SCREEN) ===

VERIFICAÇÃO DE PONTOS ANALISADOS
Cluster 0.0: 0/30 pontos já analisados (total disponível: 191)
Cluster 1.0: 0/30 pontos já analisados (total disponível: 57)
Cluster 2.0: 0/30 pontos já analisados (total disponível: 71)
Cluster 3.0: 0/30 pontos já analisados (total disponível: 73)

📊 Cluster 0.0:
   - Já analisados: 0
   - Restantes necessários: 30
   - Disponíveis para análise: 191
   - Selecionados agora: 30

📊 Cluster 2.0:
   - Já analisados: 0
   - Restantes necessários: 30
   - Disponíveis para análise: 71
   - Selecionados agora: 30

📊 Cluster 3.0:
   - Já analisados: 0
   - Restantes necessários: 30
   - Disponíveis para análise: 73
   - Selecionados agora: 30

📊 Cluster 1.0:
   - Já analisados: 0
   - Restantes necessários: 30
   - Disponíveis para análise: 57
   - Selecionados agora: 30

TOTAL DE NOVOS PONTOS SELECIONADOS: 120

Amostra de Morris gerada com 5200 pontos.

EXECUTANDO 120 ANÁLISES


[░░░░░░░░░░░░░░░░░░░░░

,mu_star,sigma,mu
parameter,,,
DVSEND,3.904622e+06,5.209466e+06,3.904622e+06
AMAXTB200,2.060856e+06,3.376910e+06,2.060856e+06
TMPFTB020,1.789259e+06,2.848556e+06,1.789259e+06
AMAXTB082,1.637658e+06,2.936554e+06,1.637658e+06
TMPFTB035,1.508118e+06,2.831861e+06,1.508118e+06
SLATB100,1.367651e+06,2.689018e+06,1.367427e+06
SPAN,1.149010e+06,2.373110e+06,1.147852e+06
TSUM2,1.115461e+06,2.281831e+06,1.090439e+06
TDWI,1.113572e+06,2.960850e+06,1.037422e+06


## Consolidar ranking por cluster e salvar para a etapa de Optimization


In [4]:
path_to_results = os.path.join(paths['RESULTS'], "SA_MORRIS_cluster*_point*.csv")
files = glob.glob(path_to_results)

dfs = [pd.read_csv(f) for f in files]
df_combined = pd.concat(dfs, ignore_index=True).round(4)

df_combined = (
    df_combined.groupby(['cluster_id', 'parameter'])[['mu_star', 'sigma']]
    .mean()
    .reset_index()
    .sort_values(by='mu_star', ascending=False)
)

ranking_por_cluster = {}
for cluster_id in df_combined['cluster_id'].unique():
    df_cluster = (
        df_combined[(df_combined['cluster_id'] == cluster_id) & (df_combined['mu_star'] > 0)]
        .sort_values(by='mu_star', ascending=False)
        .reset_index(drop=True)
    )
    df_cluster['Rank'] = df_cluster.index + 1
    ranking_por_cluster[str(cluster_id)] = df_cluster[['Rank', 'parameter', 'mu_star', 'sigma']].to_dict('records')

ranking_json_path = os.path.join(paths['RESULTS'], 'SA_ranking_by_cluster.json')
with open(ranking_json_path, 'w') as f:
    json.dump(ranking_por_cluster, f, indent=2)

print(f"Ranking por cluster salvo em {ranking_json_path}")
for cluster_id, ranking in ranking_por_cluster.items():
    top5 = [r['parameter'] for r in ranking[:5]]
    print(f"Cluster {cluster_id}: top 5 = {top5}")


Ranking por cluster salvo em d:\_py\AgroIA_prod\output\soja_pr\Sensitivity Analysis\SA_ranking_by_cluster.json
Cluster 2.0: top 5 = ['DVSEND', 'TMPFTB020', 'AMAXTB200', 'AMAXTB082', 'SLATB100']
Cluster 1.0: top 5 = ['DVSEND', 'AMAXTB200', 'TMPFTB020', 'AMAXTB082', 'TMPFTB035']
Cluster 0.0: top 5 = ['DVSEND', 'AMAXTB200', 'TMPFTB035', 'AMAXTB082', 'TMPFTB020']
Cluster 3.0: top 5 = ['DVSEND', 'AMAXTB200', 'TMPFTB035', 'AMAXTB082', 'SLATB100']
